# OCR + Vision Pipeline

Advanced PDF extraction pipeline that handles:
- **Native text** extraction (fast, for digital PDFs)
- **Vision OCR** for scanned pages (using Gemini Vision API)
- **Chart/Figure description** using Vision LLM
- **Structured Markdown** output ready for RAG ingestion

Pipeline: `PDF → PyMuPDF → Vision OCR (scanned pages) → Image Description → Markdown → Chunk → Embed → Store`

---

In [1]:
import sys
import os
from pathlib import Path
from _setup import PROJECT_ROOT

print(f'Project root: {PROJECT_ROOT}')
print(f"GEMINI_API_KEY set: {'[OK]' if os.getenv('GEMINI_API_KEY') else '[MISSING] (required for Vision OCR)'}")

Project root: C:\Users\lenovo\Desktop\Pfe 2026\financial-advisor
GEMINI_API_KEY set: [OK]


## 1. Basic PDF Text Extraction (No OCR)

Fast extraction using PyMuPDF for digitally-created PDFs.

In [2]:
import fitz  # PyMuPDF

# Point to any PDF in the project
PDF_PATH = str(PROJECT_ROOT / "data" / "collected_data" / "documents" / "ammc" / "ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf")

doc = fitz.open(PDF_PATH)
print(f"PDF: {Path(PDF_PATH).name}")
print(f"Pages: {len(doc)}")
print(f"Title: {doc.metadata.get('title', 'Unknown')}")
print(f"Author: {doc.metadata.get('author', 'Unknown')}")

# Extract text from first page
page = doc[0]
text = page.get_text("text")
print(f"\n--- Page 1 Text ({len(text)} chars) ---")
print(text[:500])
doc.close()

PDF: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf
Pages: 22
Title: 
Author: 

--- Page 1 Text (40 chars) ---
Guide pédagogique
COPRENDRE LES OPCV 



## 2. Detect Scanned vs Digital Pages

In [3]:
doc = fitz.open(PDF_PATH)

print(f"Page Analysis for: {Path(PDF_PATH).name}")
print(f"{'Page':<6} {'Text Chars':<12} {'Images':<8} {'Type'}")
print("-" * 50)

for i in range(min(len(doc), 10)):  # First 10 pages
    page = doc[i]
    text = page.get_text("text").strip()
    images = page.get_images(full=True)
    
    # If very little text, it's likely scanned
    page_type = "SCANNED" if len(text) < 50 else "DIGITAL"
    print(f"{i+1:<6} {len(text):<12} {len(images):<8} {page_type}")

doc.close()

Page Analysis for: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf
Page   Text Chars   Images   Type
--------------------------------------------------
1      38           0        SCANNED
2      0            0        SCANNED
3      776          0        DIGITAL
4      0            0        SCANNED
5      2214         0        DIGITAL
6      2813         0        DIGITAL
7      3383         0        DIGITAL
8      3473         0        DIGITAL
9      2211         0        DIGITAL
10     2480         0        DIGITAL


## 3. Run Full OCR Pipeline

Uses Vision AI for scanned pages and chart descriptions.

Requires `GEMINI_API_KEY` in `.env`

In [4]:
from OCR_Processing.pdf_extractor import AdvancedPDFPipeline

# Initialize the pipeline
pipeline = AdvancedPDFPipeline()

print(f"Processing: {Path(PDF_PATH).name}")
print("This may take a few minutes for large PDFs with scanned pages...\n")

# Run the pipeline
output_path = pipeline.process_pdf(PDF_PATH)

print(f"\n[OK] Output saved to: {output_path}")

2026-07-02 14:47:44,368 - INFO - Starting advanced extraction on: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf
2026-07-02 14:47:44,371 - INFO - Processing page 1/22
2026-07-02 14:47:44,383 - INFO - Page 1 has limited native text. Using Vision OCR.


Processing: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf
This may take a few minutes for large PDFs with scanned pages...



2026-07-02 14:47:46,598 - ERROR - Vision API Error: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions
2026-07-02 14:47:46,598 - INFO - Processing page 2/22
2026-07-02 14:47:46,598 - INFO - Page 2 has limited native text. Using Vision OCR.
2026-07-02 14:47:47,143 - ERROR - Vision API Error: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions
2026-07-02 14:47:47,148 - INFO - Processing page 3/22
2026-07-02 14:47:47,159 - INFO - Processing page 4/22
2026-07-02 14:47:47,162 - INFO - Page 4 has limited native text. Using Vision OCR.
2026-07-02 14:47:47,688 - ERROR - Vision API Error: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions
2026-07-02 14:47:47,697 - INFO - Processing page 5/22
2026-07-02 14:47:47,699 - INFO - Processing page 6/22
2026-07-02 14:47:47,714 - INFO - Processing page 7/22
2026-07-02 14:47:47,718 - INFO - Processing page 8/22
2026-07-02 14:47:47,724 - INFO


[OK] Output saved to: OCR_Processing\output\ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794_extracted.md


In [5]:
# Read and display the extracted Markdown
md_content = Path(output_path).read_text(encoding="utf-8")

print(f"Extracted Markdown: {len(md_content)} chars\n")
print("--- First 2000 chars ---")
print(md_content[:2000])

Extracted Markdown: 37944 chars

--- First 2000 chars ---
# Document Metadata: ammc_AMMC_Guide_pédagogique_Comprendre_les_OPCVMpdf_a7bf5794.pdf

- **Pages**: 22
- **Title**: 
- **Author**: 

---

## Page 1

*(Scanned Page / OCR Extraction)*

*[Error analyzing image: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions]*


---

## Page 2

*(Scanned Page / OCR Extraction)*

*[Error analyzing image: 402 Client Error: Payment Required for url: https://openrouter.ai/api/v1/chat/completions]*


---

## Page 3

Guide pédagogique : Comprendre les OPCVM
AMMC 2018
03
Dans le cadre de sa mission d’éducation financière, l’Autorité 
Marocaine du Marché des Capitaux élabore une série d’outils 
didactiques visant à faciliter la compréhension par les 
investisseurs et le public en général des instruments financiers 
disponibles sur le marché des capitaux.
A ce titre, le guide pédagogique «Comprendre les OPCVM» 
présente les principaux concepts liés à l’investisseme

## 4. Ingest OCR Output into ChromaDB

Take the extracted Markdown and run it through the ingestion pipeline.

In [7]:
from ingestion import IngestionPipeline
from ingestion.chunkers.table_aware import TableAwareChunker

# Use the table-aware chunker for OCR output (preserves table structures)
ingestion_pipeline = IngestionPipeline()

# Create records from the markdown
records = [(md_content, {
    "source_file": Path(PDF_PATH).name,
    "extraction_method": "ocr_vision",
    "ocr": "true",
})]

# Use TableAwareChunker for better preservation of financial tables
chunker = TableAwareChunker(chunk_size=800, chunk_overlap=100)

result = ingestion_pipeline.ingest_records(records, chunker=chunker)

print(f"\n[OK] Ingestion Result:")
print(f"  Documents: {result['documents']}")
print(f"  Chunks:    {result['chunks']}")
print(f"  Stored:    {result['stored']}")

2026-07-02 14:55:50,479 - INFO - Embedder: loading model 'BAAI/bge-m3'
2026-07-02 14:55:50,644 - INFO - Use pytorch device_name: cpu
2026-07-02 14:55:50,644 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2026-07-02 14:55:50,873 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:55:50,899 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/modules.json "HTTP/1.1 200 OK"
2026-07-02 14:55:51,091 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:55:51,120 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-07-02 14:55:51,279 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/b

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

2026-07-02 14:55:53,103 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/discussions?p=0 "HTTP/1.1 200 OK"
2026-07-02 14:55:53,258 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:55:53,279 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/commits/refs%2Fpr%2F130 "HTTP/1.1 200 OK"
2026-07-02 14:55:53,288 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-m3/5617a9f61b028005a4858fdac845db406aefb181/config.json "HTTP/1.1 200 OK"
2026-07-02 14:55:53,419 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/refs%2Fpr%2F130/model.safetensors.index.json "HTTP/1.1 404 Not Found"
2026-07-02 14:55:53,431 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-07-02 14:55:53,475 - INFO - HTTP Request: HEAD https://huggingface.co/api/r

MetadataError: Metadata is missing required fields: {'doc_type', 'source'}. Got keys: {'extraction_method', 'ocr', 'source_file'}

## 5. Compare: Basic Loader vs OCR Pipeline

In [ ]:
from ingestion.loaders.pdf_loader import PDFLoader

# Basic extraction
basic_loader = PDFLoader()
basic_records = basic_loader.load(PDF_PATH)

basic_text = basic_records[0][0] if basic_records else ""

print("Comparison: Basic Loader vs OCR Pipeline")
print("=" * 50)
print(f"  Basic Loader:  {len(basic_text):,} chars extracted")
print(f"  OCR Pipeline:  {len(md_content):,} chars extracted")
print(f"  Improvement:   {((len(md_content) - len(basic_text)) / max(len(basic_text), 1) * 100):.1f}%")
print(f"\n  OCR adds: structured headings, table preservation, chart descriptions")